# Tutorial 7 — Optical Parametric Amplification and Oscillation

We build up from single-pass OPA to a self-starting synchronously-pumped OPO,
using the same TFLN waveguide as the SHG tutorials.

The chi(2) NEE naturally handles all three-wave mixing processes
(SHG, DFG, OPA) in a single equation — we just need to set up the
right input conditions.

**Physics recap:**
- The QPM poling (pp=5.18 um) phase-matches 2 um ↔ 1 um
- SHG: pump at 2 um → SH at 1 um (Tutorial 5)
- OPA: pump at 1 um + seed at 2 um → amplified signal at 2 um
- OPO: pump at 1 um + vacuum noise → self-starting signal at 2 um (with cavity)

In [ ]:
import numpy as np
from numpy.fft import fft, ifft, fftshift, fftfreq
import matplotlib.pyplot as plt
import time

import snow.pulses as pulses
import snow.waveguides as waveguides

from scipy.constants import pi, c
nm = 1e-9
um = 1e-6
mm = 1e-3
ps = 1e-12
fs = 1e-15
MHz = 1e6
THz = 1e12
pJ = 1e-12
uW = 1e-6
mW = 1e-3

plt.rcParams.update({'font.size': 14})

## Grid and Waveguide

In [ ]:
# Broadband grid spanning both pump (1um) and signal (2um)
lam_start = 800*nm
lam_stop = 3*um
f_max = c/lam_start
f_min = c/lam_stop
BW = f_max - f_min
N = 2**10
dt = 1/BW
T = N/BW
t = -T/2 + np.arange(0, T, step=dt)
f = fftfreq(N, dt)
f_ref = (f_max + f_min)/2
f_abs = f + f_ref
wl = c/f_abs
frep = 250*MHz

# Waveguide
pp = 5.18*um
L = 4*mm

def make_wg(L_val=L):
    wg = waveguides.waveguide(w_top=1800*nm, h_thinfilm=700*nm, h_etch=350*nm,
                              tf_material='LN_MgO_e', box_material='SiO2', clad_material='Air')
    wg.add_poling(lambda z: np.sign(np.cos(z*2*pi/pp)))
    wg.set_nonlinear_coeffs(N=1, X0=1.1e-12)
    wg.set_length(L_val)
    wg.set_loss(0)
    return wg

wg = make_wg()
v_ref = 1/wg.beta1(1.5*um)  # frame centered between pump and signal

# Phase-matching check
n_pump = wg.neff(np.array([1*um]))
n_sig  = wg.neff(np.array([2*um]))
pp_calc = 1*um / (n_pump - n_sig)
print(f'Phase-matching poling period: {pp_calc[0]/um:.3f} um (using {pp/um:.3f} um)')
print(f'GVM: {(wg.beta1(1*um) - wg.beta1(2*um))/(fs/mm):.1f} fs/mm')

## 1. Single-pass OPA (seeded)

Pump at 1 um, weak seed at 2 um.  The chi(2) interaction amplifies the seed.
We combine both into a single input pulse on the same grid.

In [ ]:
# Pump: 100 fs sech at 1 um
pump_opa = pulses.sech_pulse(t, 100*fs, f_ref=f_ref, f0=c/(1*um),
                             Pavg=100*uW, Npwr_dB=200, frep=frep)

# Seed: weak signal at 2 um (40 dB below pump)
seed = pulses.sech_pulse(t, 100*fs, f_ref=f_ref, f0=c/(2*um),
                         Pavg=0.01*uW, Npwr_dB=200, frep=frep)

# Combined input
input_opa = pump_opa + seed

print(f'Pump energy:  {pump_opa.energy_td()/pJ:.3f} pJ')
print(f'Seed energy:  {seed.energy_td()/pJ:.6f} pJ')
print(f'Total input:  {input_opa.energy_td()/pJ:.3f} pJ')

# Propagate
out_opa, _ = wg.propagate_NEE(input_opa, v_ref=v_ref, verbose=False)

# Analyze: filter pump and signal bands
sig_in = seed.apply_filter(c/(2*um), 40*THz)
sig_out = out_opa.apply_filter(c/(2*um), 40*THz)
pump_out = out_opa.apply_filter(c/(1*um), 40*THz)

gain = sig_out.energy_td() / sig_in.energy_td()
print(f'\nSignal gain: {gain:.1f}x ({10*np.log10(gain):.1f} dB)')
print(f'Pump depletion: {1 - pump_out.energy_td()/pump_opa.energy_td():.1%}')

# Plot
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5), tight_layout=True)
input_opa.plot_PSD(ax=ax1, f_unit='um')
out_opa.plot_PSD(ax=ax1, f_unit='um')
ax1.get_lines()[-2].set(color='k', linestyle='--', alpha=0.4, label='Input')
ax1.get_lines()[-1].set(color='b', linewidth=1.5, label='Output')
ax1.legend(); ax1.set_title('OPA: Pump (1um) + Seed (2um)')

input_opa.plot_magsq(ax=ax2, t_unit='ps')
out_opa.plot_magsq(ax=ax2, t_unit='ps')
ax2.get_lines()[-2].set(color='k', linestyle='--', alpha=0.4, label='Input')
ax2.get_lines()[-1].set(color='b', linewidth=1.5, label='Output')
ax2.legend(); ax2.set_title('Temporal Intensity')
plt.show()

## 2. Parametric fluorescence (unseeded)

Pump at 1 um with only vacuum noise at 2 um (no seed).
The parametric gain amplifies the quantum noise into a measurable signal.
This is the single-pass precursor to an OPO.

In [ ]:
# Pump only, with quantum noise enabled
pump_pf = pulses.sech_pulse(t, 100*fs, f_ref=f_ref, f0=c/(1*um),
                            Pavg=100*uW, Npwr_dB=200, frep=frep)

out_pf, _ = wg.propagate_NEE(pump_pf, v_ref=v_ref, verbose=False, Qnoise=True)

sig_pf = out_pf.apply_filter(c/(2*um), 40*THz)
print(f'Parametric fluorescence energy at 2um: {sig_pf.energy_td()/pJ:.6f} pJ')
print(f'(vs seeded OPA: {sig_out.energy_td()/pJ:.6f} pJ)')

fig, ax = plt.subplots(figsize=(8, 5), tight_layout=True)
pump_pf.plot_PSD(ax=ax, f_unit='um')
out_pf.plot_PSD(ax=ax, f_unit='um')
ax.get_lines()[-2].set(color='k', linestyle='--', alpha=0.4, label='Input (pump + noise)')
ax.get_lines()[-1].set(color='b', linewidth=1.5, label='Output')
ax.legend(); ax.set_title('Parametric Fluorescence (noise-seeded)')
plt.show()

## 3. Synchronously-pumped OPO

Now we add a cavity around the signal.  Each roundtrip:

1. Fresh pump pulse (1 um) enters the waveguide
2. Recycled signal (2 um) from previous roundtrip is added
3. Both propagate through the chi(2) waveguide (NEE)
4. Output is filtered to keep the signal band (~2 um)
5. Cavity feedback: losses, dispersion, outcoupling
6. Remaining signal feeds back for next roundtrip

The signal builds up from vacuum noise over many roundtrips.

In [ ]:
# OPO parameters
n_roundtrips = 80
Pavg_pump = 500*uW        # pump average power
outcoupling = 0.1          # 10% output coupler
cavity_loss = 0.05         # 5% round-trip cavity loss (besides outcoupling)
feedback_frac = 1 - outcoupling - cavity_loss  # fraction fed back

# Signal band filter
f0_signal = c/(2*um)
signal_bw = 40*THz

# Storage for monitoring build-up
sig_energy = np.zeros(n_roundtrips)
pump_depletion = np.zeros(n_roundtrips)
out_energy = np.zeros(n_roundtrips)  # outcoupled energy

# Initial signal: vacuum noise (no coherent seed)
recycled_signal = pulses.noise(t, 0)  # zero — noise comes from Qnoise

print(f'OPO: {n_roundtrips} roundtrips, Ppump={Pavg_pump/uW:.0f} uW, '
      f'outcoupling={outcoupling:.0%}, cavity loss={cavity_loss:.0%}')
print()

t_start = time.perf_counter()
for rt in range(n_roundtrips):
    # Fresh pump each roundtrip
    pump_rt = pulses.sech_pulse(t, 100*fs, f_ref=f_ref, f0=c/(1*um),
                                Pavg=Pavg_pump, Npwr_dB=200, frep=frep)

    # Combine pump + recycled signal
    # Create the recycled signal as a pulse object
    sig_pulse = pulses.pulse(t, recycled_signal, c/f_ref, frep)
    input_rt = pump_rt + sig_pulse

    # Propagate through waveguide (Qnoise seeds vacuum fluctuations)
    out_rt, _ = wg.propagate_NEE(input_rt, v_ref=v_ref, verbose=False,
                                 Qnoise=True)

    # Filter signal band
    sig_out_rt = out_rt.apply_filter(f0_signal, signal_bw)
    pump_out_rt = out_rt.apply_filter(c/(1*um), 40*THz)

    # Record
    E_sig = sig_out_rt.energy_td()
    sig_energy[rt] = E_sig
    pump_depletion[rt] = 1 - pump_out_rt.energy_td() / pump_rt.energy_td()
    out_energy[rt] = E_sig * outcoupling

    # Cavity feedback: keep signal, apply losses
    recycled_signal = sig_out_rt.a * np.sqrt(feedback_frac)

    if (rt+1) % 20 == 0 or rt == 0:
        print(f'  RT {rt+1:3d}: signal={E_sig/pJ:.4e} pJ, '
              f'pump depl={pump_depletion[rt]:.1%}')

t_opo = time.perf_counter() - t_start
print(f'\nTotal time: {t_opo:.1f}s ({t_opo/n_roundtrips:.2f}s per roundtrip)')

In [ ]:
# Plot OPO build-up
fig, axes = plt.subplots(2, 2, figsize=(14, 10), tight_layout=True)

# Signal energy vs roundtrip
ax = axes[0, 0]
ax.semilogy(np.arange(1, n_roundtrips+1), sig_energy/pJ)
ax.set_xlabel('Roundtrip'); ax.set_ylabel('Signal energy (pJ)')
ax.set_title('OPO Build-up'); ax.grid(True)

# Pump depletion vs roundtrip
ax = axes[0, 1]
ax.plot(np.arange(1, n_roundtrips+1), pump_depletion * 100)
ax.set_xlabel('Roundtrip'); ax.set_ylabel('Pump depletion (%)')
ax.set_title('Pump Depletion'); ax.grid(True)

# Output spectrum at steady state
ax = axes[1, 0]
out_rt.plot_PSD(ax=ax, f_unit='um')
pump_rt.plot_PSD(ax=ax, f_unit='um')
ax.get_lines()[-1].set(color='k', linestyle='--', alpha=0.3, label='Input pump')
ax.get_lines()[-2].set(color='b', linewidth=1.5, label='Output (last RT)')
ax.legend(); ax.set_title('Output Spectrum (last roundtrip)')

# Output temporal profile
ax = axes[1, 1]
sig_out_rt.plot_magsq(ax=ax, t_unit='ps')
ax.set_title('Signal Pulse (last roundtrip)')

plt.show()

# Steady-state summary
ss = slice(-10, None)  # last 10 roundtrips
print(f'Steady-state signal energy: {np.mean(sig_energy[ss])/pJ:.4f} pJ')
print(f'Steady-state pump depletion: {np.mean(pump_depletion[ss]):.1%}')
print(f'Outcoupled power: {np.mean(out_energy[ss])*frep/uW:.2f} uW')

## 4. OPO threshold scan

Sweep the pump power to find the OPO threshold — the minimum power
where the parametric gain exceeds the round-trip cavity loss.

In [ ]:
from IPython.display import display

P_thresh = np.array([10, 30, 50, 100, 200, 500, 1000]) * uW
n_rt_thresh = 60

plt.ioff()
fig_th, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5), tight_layout=True)
ax1.set_xlabel('Roundtrip'); ax1.set_ylabel('Signal energy (pJ)')
ax1.set_title('OPO Build-up at Different Pump Powers')
ax1.set_yscale('log'); ax1.grid(True)
ax2.set_xlabel('Pump power (uW)'); ax2.set_ylabel('Output signal power (uW)')
ax2.set_title('OPO Threshold Curve'); ax2.grid(True)
fig_handle = display(fig_th, display_id=True)
plt.ion()

ss_powers = []
colors = plt.cm.viridis(np.linspace(0.2, 0.9, len(P_thresh)))

for ip, Pp in enumerate(P_thresh):
    recycled = pulses.noise(t, 0)
    energies = np.zeros(n_rt_thresh)

    for rt in range(n_rt_thresh):
        pump = pulses.sech_pulse(t, 100*fs, f_ref=f_ref, f0=c/(1*um),
                                 Pavg=Pp, Npwr_dB=200, frep=frep)
        sig_p = pulses.pulse(t, recycled, c/f_ref, frep)
        inp = pump + sig_p
        out, _ = wg.propagate_NEE(inp, v_ref=v_ref, verbose=False, Qnoise=True)
        sig = out.apply_filter(f0_signal, signal_bw)
        energies[rt] = sig.energy_td()
        recycled = sig.a * np.sqrt(feedback_frac)

    ss_power = np.mean(energies[-10:]) * outcoupling * frep
    ss_powers.append(ss_power)

    ax1.semilogy(np.arange(1, n_rt_thresh+1), energies/pJ,
                 color=colors[ip], label=f'{Pp/uW:.0f} uW')
    ax1.legend(fontsize=9, ncol=2)
    fig_handle.update(fig_th)
    print(f'  Ppump={Pp/uW:6.0f} uW: output={ss_power/uW:.4f} uW')

ax2.plot(P_thresh/uW, np.array(ss_powers)/uW, 'ko-', markersize=8)
ax2.axhline(y=0, color='k', linewidth=0.5)
fig_handle.update(fig_th)

## Summary

We demonstrated the full progression from single-pass OPA to a self-starting
synchronously-pumped OPO, all using the same NEE solver and waveguide model.

Key observations:
- **OPA**: seeded parametric amplification shows clear signal gain
- **Parametric fluorescence**: vacuum noise is amplified to measurable levels
- **OPO build-up**: signal grows exponentially from noise, saturates at
  pump depletion
- **Threshold**: minimum pump power where round-trip gain exceeds loss

The `Qnoise=True` parameter adds half-photon-per-mode vacuum fluctuation
noise, providing the physical seed for the parametric process.